In [14]:
import geopandas as gpd
import pandas as pd

In [20]:
lakes = gpd.read_file(r'model_data\Final data\lakes_imputed_24.gpkg')
model_output = pd.read_csv(r'torch_24\trial_1064_full_dataset_susceptibility.csv')

In [53]:
import os
import pandas as pd
import geopandas as gpd
from shapely import wkt

csv_path = r"torch_24\trial_1064_full_dataset_susceptibility.csv"
gpkg_path = r"model_data\Final data\trial_1064_full_dataset_susceptibility.gpkg"
layer_name = "trial_1064_full_dataset_susceptibility"

# 1) Read as normal table
df = pd.read_csv(csv_path)

# 2) Convert WKT text to real geometry
df["geometry"] = df["geometry"].apply(lambda x: wkt.loads(x) if pd.notna(x) else None)

# 3) Build GeoDataFrame with CRS (use lakes CRS from your notebook)
gdf = gpd.GeoDataFrame(df, geometry="geometry", crs=lakes.crs)

# 4) Optional sanity check
print("Null geometry:", gdf.geometry.isna().sum())
print("Invalid geometry:", (~gdf.geometry.is_valid).sum())

# 5) If file exists and may be locked/open, remove first (or close in QGIS)
if os.path.exists(gpkg_path):
    os.remove(gpkg_path)

# 6) Write to GPKG
gdf.to_file(gpkg_path, layer=layer_name, driver="GPKG")
print("Saved:", gpkg_path)

Null geometry: 0
Invalid geometry: 8
Saved: model_data\Final data\trial_1064_full_dataset_susceptibility.gpkg


In [17]:
glof = gpd.read_file(r'model_data\shapefiles\glofs\glof_final.gpkg')

In [18]:
glof.columns

Index(['s1_GL_ID', 's1_Elev_lake', 's2_Lake_type', 'long', 'lat', 'event_date',
       'index_right', 'GLAKE_ID', 'geometry'],
      dtype='object')

In [19]:
glof

,s1_GL_ID,s1_Elev_lake,s2_Lake_type,long,lat,event_date,index_right,GLAKE_ID,geometry
0,GL093631E30357N,4447,moraine,93.631000,30.356000,26-06-2020,566.0,GL093631E30357N,POINT (-122601.111 43193.57)
1,GL094001E30829N,4737,moraine,94.000000,30.830000,29-07-2009,575.0,GL094001E30827N,POINT (-88982.036 99247.554)
2,GL364575N748796E,2562,None,74.879432,36.457073,06-01-2008,6916.0,GL364575N748796E,POINT (-1640332.564 943928.557)
3,GL086064E28078N,4624,moraine,86.064000,28.078000,05-07-2016,6917.0,GL280783N860639E,POINT (-823707.362 -190154.62)
4,GL071128E36474N,4593,moraine,71.128953,36.474746,09-09-2013,86.0,GL071128E36474N,POINT (-1940022.311 1014355.053)
...,...,...,...,...,...,...,...,...,...
69,GL091936E28802N,5206,moraine,91.937000,28.802000,26-05-1995,252.0,GL091936E28802N,POINT (-280020.044 -137714.967)
70,GL356763N771922E,4880,ice,77.190788,35.677371,20-09-2004,6913.0,GL356763N771922E,POINT (-1471116.448 814695.11)
71,GL356763N771922E,4880,ice,77.190788,35.677371,02-08-2006,6913.0,GL356763N771922E,POINT (-1471116.448 814695.11)
72,GL090166E28099N,4272,supraglacial,90.164108,28.100736,29-04-2009,NaN,GL090166E28099N,POINT (-446068.304 -213973.689)


In [21]:
model_output = gpd.read_file(r'model_data\Final data\trial_1064_full_dataset_susceptibility.gpkg')

In [25]:
import numpy as np
import pandas as pd

# --- Config ---
TOTAL_ROWS = 1500
TARGET_EVENT_LAKE_NEGATIVES = 200
RANDOM_STATE = 42
DATE_START = pd.Timestamp('1975-01-01')
DATE_END = pd.Timestamp('2025-12-31')

# --- Required columns ---
if 'glof_happened' not in model_output.columns:
    raise KeyError("`model_output` must contain `glof_happened`.")
if 'GLAKE_ID' not in model_output.columns:
    raise KeyError("`model_output` must contain `GLAKE_ID`.")
if 'GLAKE_ID' not in glof.columns:
    raise KeyError("`glof` must contain `GLAKE_ID`.")

# Detect event-date column in glof
event_date_candidates = ['event_date', 'glof_event', 'event_dt', 'date']
event_date_col = next((c for c in event_date_candidates if c in glof.columns), None)
if event_date_col is None:
    raise KeyError(
        "Could not find event date column in `glof`. "
        "Expected one of: event_date, glof_event, event_dt, date"
    )

# Detect lat/long columns in glof (if present)
lat_candidates = ['lat', 'latitude', 'LAT', 'Latitude']
lon_candidates = ['long', 'lon', 'longitude', 'LONG', 'LON', 'Longitude']
lat_src = next((c for c in lat_candidates if c in glof.columns), None)
lon_src = next((c for c in lon_candidates if c in glof.columns), None)

# Normalize join IDs on both tables using GLAKE_ID as requested
model_work = model_output.copy()
glof_work = glof.copy()
model_work['_join_id'] = model_work['GLAKE_ID'].astype(str).str.strip().str.upper()
glof_work['_join_id'] = glof_work['GLAKE_ID'].astype(str).str.strip().str.upper()

# Actual event rows: ~74 event rows with the original event_date from glof
actual_event_rows = glof_work[glof_work[event_date_col].notna()].copy()
actual_event_rows = actual_event_rows.rename(columns={event_date_col: 'event_date'})
actual_event_rows['event_date'] = pd.to_datetime(actual_event_rows['event_date'], errors='coerce', dayfirst=True)
actual_event_rows = actual_event_rows[actual_event_rows['event_date'].notna()].copy()
actual_event_lake_ids = set(actual_event_rows['_join_id'].unique())

# Keep the positive lakes from model_output only where they have actual glof events
affected_lakes = model_work[
    model_work['glof_happened'].eq(1) & model_work['_join_id'].isin(actual_event_lake_ids)
].copy()
affected_lakes = affected_lakes.drop_duplicates(subset=['_join_id'], keep='first').copy()

# Keep only event rows that match a positive lake in model_output
actual_pos_df = affected_lakes.merge(
    actual_event_rows,
    on='_join_id',
    how='inner',
    suffixes=('', '_glof'),
)
actual_pos_df['source_type'] = 'actual_event'
actual_pos_df['glof_happened'] = 1

# Canonicalize coordinate column names if they exist in glof
if lat_src is not None:
    lat_col = lat_src if lat_src in actual_pos_df.columns else f'{lat_src}_glof'
    if lat_col in actual_pos_df.columns and 'lat' not in actual_pos_df.columns:
        actual_pos_df = actual_pos_df.rename(columns={lat_col: 'lat'})
if lon_src is not None:
    lon_col = lon_src if lon_src in actual_pos_df.columns else f'{lon_src}_glof'
    if lon_col in actual_pos_df.columns and 'long' not in actual_pos_df.columns:
        actual_pos_df = actual_pos_df.rename(columns={lon_col: 'long'})

# Build lake lookup for the event lakes so negative versions keep lake attributes
lake_lookup_cols = ['_join_id']
if lat_src is not None:
    lake_lookup_cols.append(lat_src)
if lon_src is not None:
    lake_lookup_cols.append(lon_src)
lake_lookup = glof_work[glof_work['_join_id'].isin(actual_event_lake_ids)][lake_lookup_cols].drop_duplicates(subset=['_join_id']).copy()
if lat_src is not None and lat_src in lake_lookup.columns:
    lake_lookup = lake_lookup.rename(columns={lat_src: 'lat'})
if lon_src is not None and lon_src in lake_lookup.columns:
    lake_lookup = lake_lookup.rename(columns={lon_src: 'long'})

# Random dates for negatives
all_random_dates = pd.date_range(DATE_START, DATE_END, freq='D')

def assign_random_dates(frame, seed_offset=0):
    if len(frame) == 0:
        return frame
    local_rng = np.random.default_rng(RANDOM_STATE + seed_offset)
    frame = frame.copy()
    frame['event_date'] = pd.to_datetime(
        local_rng.choice(all_random_dates, size=len(frame), replace=True)
    )
    return frame

# Negatives Part 1: Duplicated event lakes 
# We take the affected lakes, sample with replacement to hit TARGET_EVENT_LAKE_NEGATIVES, 
# and give them random dates, treating them as negatives
if len(affected_lakes) > 0:
    event_lake_negatives_df = affected_lakes.sample(
        n=TARGET_EVENT_LAKE_NEGATIVES, 
        replace=True, 
        random_state=RANDOM_STATE
    ).copy()
else:
    event_lake_negatives_df = affected_lakes.head(0).copy()

event_lake_negatives_df = event_lake_negatives_df.merge(lake_lookup, on='_join_id', how='left')
event_lake_negatives_df = assign_random_dates(event_lake_negatives_df, seed_offset=1)
event_lake_negatives_df['source_type'] = 'event_lake_negative'
event_lake_negatives_df['glof_happened'] = 0

# Negatives Part 2: Non-event lakes weighted by susceptibility
negative_pool = model_work[model_work['glof_happened'].eq(0)].copy()
negative_pool = negative_pool.drop_duplicates(subset=['_join_id'], keep='first').copy()

# Calculate how many rows we need to reach TOTAL_ROWS
current_rows = len(actual_pos_df) + len(event_lake_negatives_df)
negative_needed = max(TOTAL_ROWS - current_rows, 0)

sus_col_candidates = ['susceptibility_band', 'susceptibility', 'Susceptibility', 'sus_band']
sus_col = next((c for c in sus_col_candidates if c in negative_pool.columns), None)

if negative_needed > 0 and len(negative_pool) > 0:
    negative_needed_actual = min(negative_needed, len(negative_pool))
    if sus_col is not None:
        sus_norm = negative_pool[sus_col].astype(str).str.strip().str.lower()
        weight_map = {
            'moderate': 10.0,
            'high': 10.0,
            'very high': 15.0,
            'very_high': 15.0,
        }
        neg_weights = sus_norm.map(weight_map).fillna(1.0)
        other_negatives_df = negative_pool.sample(
            n=negative_needed_actual,
            replace=False,
            weights=neg_weights,
            random_state=RANDOM_STATE,
        ).copy()
    else:
        other_negatives_df = negative_pool.sample(
            n=negative_needed_actual,
            replace=False,
            random_state=RANDOM_STATE,
        ).copy()
else:
    other_negatives_df = negative_pool.head(0).copy()

other_negatives_df = other_negatives_df.merge(lake_lookup, on='_join_id', how='left')
other_negatives_df = assign_random_dates(other_negatives_df, seed_offset=2)
other_negatives_df['source_type'] = 'other_negative'
other_negatives_df['glof_happened'] = 0

# Make sure all frames share the same columns
all_frames = [actual_pos_df, event_lake_negatives_df, other_negatives_df]
all_columns = sorted(set().union(*(f.columns for f in all_frames)))

for i, frame in enumerate(all_frames):
    for col in all_columns:
        if col not in frame.columns:
            frame[col] = np.nan
    all_frames[i] = frame[all_columns]

final_df = pd.concat(all_frames, ignore_index=True)
final_df = final_df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print(f"Total target rows: {TOTAL_ROWS}")
print(f"Positive rows (actual events, actual dates): {len(actual_pos_df)}")
print(f"Event lakes converted to negative (random dates): {len(event_lake_negatives_df)}")
print(f"Other negative rows sampled: {len(other_negatives_df)}")
print(f"Final total rows: {len(final_df)}")
print(f"Total positive (glof_happened=1): {final_df['glof_happened'].eq(1).sum()}")
print(f"Total negative (glof_happened=0): {final_df['glof_happened'].eq(0).sum()}")
print("\nRow types:")
print(final_df['source_type'].value_counts())

final_df.head(10)

Total target rows: 1500
Positive rows (actual events, actual dates): 74
Event lakes converted to negative (random dates): 200
Other negative rows sampled: 1226
Final total rows: 1500
Total positive (glof_happened=1): 74
Total negative (glof_happened=0): 1426

Row types:
source_type
other_negative         1226
event_lake_negative     200
actual_event             74
Name: count, dtype: int64


,Elevation,G11_mean_slope_deg,GLAKE_ID,GLAKE_ID_glof,RGIId,Type,_join_id,area_1990,area_1990_km2,area_2000,...,s2_Lake_type,source_type,surface_elevation_m,susceptibility,susceptibility_band,susceptibility_prob,v_mean_2020,volume_m3,watershed_area_km2,wse_m_dam
0,4718.0,15.060656,GL092352E27777N,NaN,RGI60-15.01960,Unconnected glacial lakes,GL092352E27777N,3.708817e+04,0.037088,3.311815e+04,...,NaN,other_negative,4717.00,high,high,0.897765,8.951847,6.971179e+04,0.155,4716.500000
1,4913.0,20.293869,GL092433E29315N,NaN,RGI60-13.04733,Unconnected glacial lakes,GL092433E29315N,1.169496e+05,0.116950,1.197129e+05,...,NaN,other_negative,4907.00,very low,very low,0.000259,0.685119,4.819720e+03,0.067,4907.000000
2,4292.0,15.376351,GL096082E29338N,NaN,RGI60-15.12157,Unconnected glacial lakes,GL096082E29338N,2.940664e+05,0.294066,2.766917e+05,...,NaN,other_negative,4280.00,high,high,0.982959,1.109303,2.870450e+06,0.078,4280.000000
3,3850.0,33.257790,GL073473E36623N,NaN,RGI60-14.00097,Unconnected glacial lakes,GL073473E36623N,3.874448e+04,0.038744,3.579332e+04,...,NaN,other_negative,3850.61,very high,very high,0.999930,2.490637,1.099328e+05,0.043,3848.531982
4,5589.0,20.831761,GL082232E30223N,NaN,RGI60-13.26985,Pro-glacial lakes,GL082232E30223N,2.674425e+05,0.267443,3.391112e+05,...,NaN,other_negative,5583.57,very high,very high,0.997755,0.805260,3.235007e+06,0.088,5582.877309
5,5279.0,27.258285,GL085635E28494N,NaN,RGI60-15.10290,Pro-glacial lakes,GL085635E28494N,4.910623e+06,4.910623,4.945633e+06,...,NaN,other_negative,5280.50,very high,very high,0.999965,6.851459,1.627388e+08,0.071,5280.500089
6,5308.0,26.060836,GL096823E29865N,NaN,RGI60-15.08532,Unconnected glacial lakes,GL096823E29865N,3.055954e+04,0.030560,2.629430e+04,...,NaN,other_negative,5301.92,very low,very low,0.002083,0.544696,3.014060e+03,0.620,5302.701660
7,5337.0,18.784246,GL086548E28202N,NaN,RGI60-15.09798,Unconnected glacial lakes,GL086548E28202N,6.474559e+04,0.064746,5.486209e+04,...,NaN,other_negative,5332.55,very high,very high,0.999804,1.000466,8.370547e+04,0.199,5332.962619
8,5469.0,13.743913,GL090186E28248N,NaN,RGI60-13.26237,Pro-glacial lakes,GL090186E28248N,4.708641e+05,0.470864,5.437884e+05,...,NaN,other_negative,5450.75,very high,very high,0.999825,14.009385,4.123096e+06,8.285,5450.194648
9,4751.0,20.615834,GL073196E36026N,NaN,RGI60-14.21343,Unconnected glacial lakes,GL073196E36026N,3.713035e+04,0.037107,3.391784e+04,...,NaN,event_lake_negative,4730.62,very high,very high,0.999970,0.459593,9.987096e+04,0.095,4727.838379


In [32]:
# Validation check for date matching
print("=== Validation: Date Matching ===")

# 1. Check actual events
actual_pos = final_df[final_df['source_type'] == 'actual_event']
print(f"Actual positive events with exact original dates: {len(actual_pos)}")

# 2. Check event lake negatives (did any random date accidentally match the true event date?)
event_neg = final_df[final_df['source_type'] == 'event_lake_negative'].copy()
# Bring in the true dates for these lakes to compare
overlap_check = event_neg.merge(
    actual_event_rows[['_join_id', 'event_date']], 
    on='_join_id', 
    suffixes=('_random', '_true')
)

accidental_matches = overlap_check[overlap_check['event_date_random'] == overlap_check['event_date_true']]

print(f"Total event_lake_negative rows: {len(event_neg)}")
print(f"Accidental exact date matches for negatives: {len(accidental_matches)}")
if len(accidental_matches) > 0:
    print("Warning: Some random dates exactly matched the actual GLOF date for that lake!")
else:
    print("Success: All generated negative dates differ from the actual GLOF dates for those lakes.")


=== Validation: Date Matching ===
Actual positive events with exact original dates: 74
Total event_lake_negative rows: 200
Accidental exact date matches for negatives: 0
Success: All generated negative dates differ from the actual GLOF dates for those lakes.


In [29]:
final_df.columns

Index(['Elevation', 'G11_mean_slope_deg', 'GLAKE_ID', 'GLAKE_ID_glof', 'RGIId',
       'Type', '_join_id', 'area_1990', 'area_1990_km2', 'area_2000',
       'area_2000_km2', 'area_2010', 'area_2010_km2', 'area_2015',
       'area_2015_km2', 'area_2020', 'area_2020_km2',
       'area_cagr_pct_1990_2000', 'area_cagr_pct_2000_2010',
       'area_cagr_pct_2010_2015', 'area_cagr_pct_2015_2020',
       'area_cagr_total_pct', 'area_cv_across_years', 'area_exp_1990_2000',
       'area_exp_2000_2010', 'area_exp_2010_2015', 'area_exp_2015_2020',
       'area_exp_pct_1990_2000', 'area_exp_pct_2000_2010',
       'area_exp_pct_2010_2015', 'area_exp_pct_2015_2020', 'area_exp_total',
       'area_exp_total_pct', 'area_m2', 'area_perimeter_ratio',
       'area_shrink_1990_2000', 'area_shrink_2000_2010',
       'area_shrink_2010_2015', 'area_shrink_2015_2020', 'area_shrink_total',
       'area_std_across_years', 'compactness', 'contact_m', 'dam_crest_m',
       'dam_height_m', 'dam_metrics_reliable', '

In [35]:
final_df.to_file(r'model_data\Final data\stage_2_final_dataset3.gpkg', driver='GPKG')

In [33]:
final_df.to_csv(r'model_data\Final data\stage_2_final_dataset3.csv', index=False)

In [56]:
import pandas as pd
import geopandas as gpd
from shapely import wkt
from shapely.geometry import Point

# 1) Build GeoDataFrame for model_output with WKT geometry
model_gdf = model_output.copy()
model_gdf["geometry"] = model_gdf["geometry"].apply(lambda x: wkt.loads(x) if pd.notna(x) else None)
model_gdf = gpd.GeoDataFrame(model_gdf, geometry="geometry", crs=lakes.crs)

print(f"Model geometry CRS: {model_gdf.crs}")
print(f"Lakes CRS: {lakes.crs}")

# 2) Build GeoDataFrame for glof with point geometry from lat/long (WGS84)
glof_gdf = glof.copy()
glof_gdf["geometry"] = glof_gdf.apply(lambda row: Point(row["long"], row["lat"]) if pd.notna(row["long"]) and pd.notna(row["lat"]) else None, axis=1)
glof_gdf = gpd.GeoDataFrame(glof_gdf, geometry="geometry", crs="EPSG:4326")

# 3) Reproject glof to match model_gdf CRS
if glof_gdf.crs != model_gdf.crs:
    glof_gdf = glof_gdf.to_crs(model_gdf.crs)
    print(f"Reprojected glof to CRS: {glof_gdf.crs}")

# 4) Spatial join: find which model polygons each glof point intersects
spatial_join = gpd.sjoin(glof_gdf, model_gdf[["GLAKE_ID", "geometry"]], how="left", predicate="intersects")

# 5) Update glof['s1_GL_ID'] where intersection found
for idx, row in spatial_join.iterrows():
    if pd.notna(row["GLAKE_ID"]):
        glof.loc[idx, "s1_GL_ID"] = row["GLAKE_ID"]

# 6) Verify
print("\nUpdated glof['s1_GL_ID'] where overlapping with model_output polygons")
print(f"Rows with updated ID: {spatial_join['GLAKE_ID'].notna().sum()}")
print(f"Rows unchanged: {spatial_join['GLAKE_ID'].isna().sum()}")
print("\nSample updated IDs:")
print(glof[["s1_GL_ID", "lat", "long"]].head(10))


Model geometry CRS: PROJCS["Asia_North_Albers_Equal_Area_Conic",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",30],PARAMETER["longitude_of_center",95],PARAMETER["standard_parallel_1",15],PARAMETER["standard_parallel_2",65],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["ESRI","102025"]]
Lakes CRS: PROJCS["Asia_North_Albers_Equal_Area_Conic",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Al

In [133]:
final_df.to_csv(r'model_data\Final data\stage_2_final_dataset2.csv')

In [107]:
final_df['geometry'] = final_df['geometry'].apply(lambda x: wkt.loads(x) if pd.notna(x) else None)

In [134]:
gdf1 = gpd.GeoDataFrame(final_df , geometry = final_df.geometry, crs = 'epsg:4326')

In [135]:
gdf1.to_file(r'model_data\Final data\stage_2_final_dataset2.gpkg', driver='GPKG')

In [129]:
import random
from datetime import datetime, timedelta

def get_random_date(start, end):
    # Calculate the range of days
    delta = end - start
    random_days = random.randrange(delta.days)
    # Add random days to start date
    return start + timedelta(days=random_days)

# Example Usage
start_date = datetime(1975, 1, 1)
end_date = datetime(2025, 12, 31)
print(get_random_date(start_date, end_date))


2008-04-30 00:00:00


In [ ]:
final_df['event_date'] = final_df['event_date'].apply(lambda x: get_random_date(start_date, end_date) if pd.isna(x) else x)

In [ ]:
import geopandas as gpd
import pandas as pd
final_df = gpd.read_file(r'model_data\Final data\stage_2_final_dataset3.gpkg')

In [1]:
import polars as pl
era5 = pl.scan_parquet(r'era5_data\era5_master_dataset.parquet')

In [2]:
import duckdb
import time

input_path = 'era5_data\era5_master_dataset.parquet'
output_path = 'era5_data\era5_master_dataset_wkt.parquet'

print("Initializing DuckDB and loading spatial extension...")
# We use a file-backed connection instead of in-memory to keep RAM usage low
con = duckdb.connect('era5_processing.db')
con.execute("INSTALL spatial;")
con.execute("LOAD spatial;")

print("Calculating statistics natively...")
# Fetch counts efficiently directly from the parquet file metadata
stats = con.execute(f"""
    SELECT 
        COUNT(*) as total_count,
        COUNT(".geo") as non_null_count
    FROM read_parquet('{input_path}')
""").fetchone()

total_count = stats[0]
non_null_count = stats[1]
null_count = total_count - non_null_count

print(f"Total Rows: {total_count}")
print(f"Null Geometries: {null_count}")
print(f"Converting {non_null_count} JSON geometries to WKT...")

# Start the timer
start_time = time.time()

# The core operation: Read, transform, and write simultaneously using all CPU cores
con.execute(f"""
    COPY (
        SELECT 
            * EXCLUDE (".geo"),
            ST_AsText(ST_GeomFromGeoJSON(".geo")) AS ".geo"
        FROM read_parquet('{input_path}')
    ) TO '{output_path}' (FORMAT PARQUET);
""")

end_time = time.time()
print(f"Conversion complete! Saved to: {output_path}")
print(f"Time taken: {round((end_time - start_time) / 60, 2)} minutes")

print("\nExtracting a sample WKT...")
sample = con.execute(f"""
    SELECT ".geo" 
    FROM read_parquet('{output_path}') 
    WHERE ".geo" IS NOT NULL 
    LIMIT 1
""").fetchone()

print(f"Sample WKT: {sample[0][:100] if sample and sample[0] else 'None'}")

# Clean up
con.close()

<>:4: SyntaxWarning: invalid escape sequence '\e'
<>:5: SyntaxWarning: invalid escape sequence '\e'
<>:4: SyntaxWarning: invalid escape sequence '\e'
<>:5: SyntaxWarning: invalid escape sequence '\e'
C:\Users\rubel\AppData\Local\Temp\ipykernel_19348\3273001746.py:4: SyntaxWarning: invalid escape sequence '\e'
  input_path = 'era5_data\era5_master_dataset.parquet'
C:\Users\rubel\AppData\Local\Temp\ipykernel_19348\3273001746.py:5: SyntaxWarning: invalid escape sequence '\e'
  output_path = 'era5_data\era5_master_dataset_wkt.parquet'


Initializing DuckDB and loading spatial extension...
Calculating statistics natively...
Total Rows: 128961644
Null Geometries: 0
Converting 128961644 JSON geometries to WKT...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Conversion complete! Saved to: era5_data\era5_master_dataset_wkt.parquet
Time taken: 2.26 minutes

Extracting a sample WKT...
Sample WKT: POINT (77.1921844792466 35.676339303574714)


In [3]:
era_wkt = pl.scan_parquet(r'era5_data\era5_master_dataset_wkt.parquet')
era_wkt.slice(0, 5).collect()

system:index,date,dewpoint_temperature_2m,fid,snow_depth_water_equivalent,snowfall_sum,snowmelt_sum,sub_surface_runoff_sum,surface_pressure,surface_runoff_sum,temperature_2m,total_precipitation_sum,.geo
str,str,f64,i64,f64,f64,f64,f64,f64,f64,f64,f64,str
"""19750101_00000000000000001b02""","""1975-01-01""",236.546048,6915,4.527558,0.000054,0.0,0.0,50793.47998,0.0,240.007981,0.000055,"""POINT (77.1921844792466 35.676…"
"""19750101_00000000000000001b03""","""1975-01-01""",238.051745,6916,0.111,0.000022,0.0,0.0,57411.896647,0.0,241.697027,0.000023,"""POINT (73.89654801821183 36.83…"
"""19750101_00000000000000001b04""","""1975-01-01""",258.672676,6917,0.045206,0.001525,0.0,0.000742,58844.521647,0.0,261.863287,0.001527,"""POINT (91.80682250694977 27.82…"
"""19750101_00000000000000001b05""","""1975-01-01""",243.200915,6918,0.132182,0.000047,0.0,0.0,64401.10498,0.0,247.478033,0.000048,"""POINT (74.87963528979537 36.45…"
"""19750101_00000000000000001b06""","""1975-01-01""",258.220121,6919,0.04256,0.022565,0.0,0.000505,57425.896647,0.0,261.154384,0.022609,"""POINT (86.06389050407549 28.07…"


In [4]:
era_wkt.select(pl.col(".geo")).slice(0, 5).collect()

.geo
str
"""POINT (77.1921844792466 35.676…"
"""POINT (73.89654801821183 36.83…"
"""POINT (91.80682250694977 27.82…"
"""POINT (74.87963528979537 36.45…"
"""POINT (86.06389050407549 28.07…"


In [39]:
unique_geo = era_wkt.select(pl.col(".geo")).unique().collect()

In [40]:
unique_geo

.geo
str
"""POINT (97.06652280791175 29.48…"
"""POINT (97.0421302485771 29.478…"
"""POINT (82.60931308127017 29.99…"
"""POINT (96.918061774106 28.9648…"
"""POINT (90.40665296687318 28.01…"
…
"""POINT (82.41278713557739 30.03…"
"""POINT (85.87143781879955 28.19…"
"""POINT (70.17764040751254 36.03…"


In [10]:
import geopandas as gpd
final_df = gpd.read_file(r'model_data\Final data\trial_1064_full_dataset_susceptibility.gpkg')

In [ ]:
import geopandas as gpd
from shapely import wkt
import polars as pl
import pandas as pd

# 1. Fetch unique WKTs directly from the saved parquet
unique_geos_df = pl.scan_parquet(r'era5_data\era5_master_dataset_wkt.parquet').select(pl.col(".geo")).unique().collect()

# Convert string texts to true Shapely geometries first
geometry_list = [wkt.loads(x) if pd.notna(x) else None for x in unique_geos_df['.geo'].to_list()]

# Build the unique_geos GeoDataFrame
unique_geos_gdf = gpd.GeoDataFrame(
    {'geometry': geometry_list},
    geometry='geometry',
    crs='EPSG:4326'
)
unique_geos_gdf = unique_geos_gdf.dropna(subset=['geometry']).reset_index(drop=True)


print("=== Checking Spatial Overlay ===")
# 2. Ensure final_df is a GeoDataFrame with the correct projected CRS (meters)
if not isinstance(final_df, gpd.GeoDataFrame):
    final_gdf = gpd.GeoDataFrame(final_df, geometry='geometry')
else:
    final_gdf = final_df.copy()

# Drop 'index_right' or 'index_left' if they exist from a previous merge
drop_cols = [c for c in final_gdf.columns if c in ['index_right', 'index_left']]
if drop_cols:
    final_gdf = final_gdf.drop(columns=drop_cols)

# Set CRS for final_gdf if missing based on lakes.crs
if final_gdf.crs is None:
    final_gdf.set_crs(lakes.crs, inplace=True)

print(f"final_df CRS: {final_gdf.crs.name if final_gdf.crs else 'None'}")
print(f"unique_geos_gdf CRS: {unique_geos_gdf.crs.name if unique_geos_gdf.crs else 'None'}")

# 3. Reproject unique_geos_gdf (degrees) to match final_gdf (meters)
unique_geos_reproj = unique_geos_gdf.to_crs(final_gdf.crs)

# 4. Check overlaps using a spatial join
overlap_sjoin = gpd.sjoin(final_gdf, unique_geos_reproj, how='inner', predicate='intersects')

# 5. Display Results
matched_final_count = overlap_sjoin.index.nunique()
coverage_pct = (matched_final_count / len(final_gdf)) * 100

print("\n--- Overlay Results ---")
print(f"Total rows in final_df: {len(final_gdf)}")
print(f"Total unique ERA5 coordinates: {len(unique_geos_gdf)}")
print(f"Rows in final_df intersected exactly by an ERA5 geospatial point/polygon: {matched_final_count} ({coverage_pct:.2f}%)")

if matched_final_count == len(final_gdf):
    print("Success: All features in final_df have at least one intersecting unique_geo point!")
else:
    print(f"Warning: {len(final_gdf) - matched_final_count} rows in final_df DO NOT cleanly intersect. They are disjoint.")

# 6. Show distance to nearest for records that DO NOT intersect
unmatched = final_gdf[~final_gdf.index.isin(overlap_sjoin.index)]
if len(unmatched) > 0:
    print(f"\nComputing distance to nearest ERA5 geometry for unmatched features...")
    # Use sjoin_nearest instead of applies for performance
    nearest_join = gpd.sjoin_nearest(unmatched, unique_geos_reproj, distance_col='dist_meters')
    final_df_node_dists = nearest_join.drop_duplicates(subset=['_join_id'])['dist_meters']
    
    print("\nDistance statistics for the un-intersected final_df features to the nearest unique geo (in meters):")
    print(final_df_node_dists.describe())


In [8]:
import geopandas as gpd

print("=== Checking Spatial Overlay using final_df Centroids ===")

# 1. Ensure final_df is a GeoDataFrame with the correct projected CRS
if not isinstance(final_df, gpd.GeoDataFrame):
    final_gdf = gpd.GeoDataFrame(final_df, geometry='geometry')
else:
    final_gdf = final_df.copy()

# Drop merge indices if they persist
drop_cols = [c for c in final_gdf.columns if c in ['index_right', 'index_left']]
if drop_cols:
    final_gdf = final_gdf.drop(columns=drop_cols)

if final_gdf.crs is None:
    final_gdf.set_crs(lakes.crs, inplace=True)

# 2. Get centroids for final_df (using the projected CRS so centroids are accurate)
final_centroids_gdf = final_gdf.copy()
final_centroids_gdf['geometry'] = final_centroids_gdf.geometry.centroid

print(f"final_centroids CRS: {final_centroids_gdf.crs.name}")
print(f"unique_geos_gdf CRS: {unique_geos_gdf.crs.name}")

# 3. Reproject unique_geos to the projected CRS for accurate distance measurement
unique_geos_reproj = unique_geos_gdf.to_crs(final_centroids_gdf.crs)

# Clean up any leftover columns from `unique_geos_gdf` if any exist
unique_geos_reproj = unique_geos_reproj[['geometry']].copy()

# 4. Check Nearest Neighbor Distances
# Since both are now points (centroids vs ERA5 points), exact intersects are highly unlikely.
# We will find the nearest ERA5 point for EVERY final_df centroid and measure the distance.
print("\nComputing distance from every final_df centroid to the nearest ERA5 point...")

nearest_join = gpd.sjoin_nearest(
    final_centroids_gdf, 
    unique_geos_reproj, 
    distance_col='dist_to_era5_meters',
    how='left'
)

# Deduplicate in case a centroid is equidistant to multiple ERA5 points
nearest_join_dedup = nearest_join.drop_duplicates(subset=['_join_id'])

distances = nearest_join_dedup['dist_to_era5_meters']

print("\n--- Centroid Distance Results ---")
print(f"Total centroids checked: {len(distances)}")
print("Distance statistics (in meters) from lake centroid to nearest ERA5 location:")
print(distances.describe())

# Check how many are within a typical grid resolution (e.g., 30km for ERA5 = 30000 meters)
within_30km = (distances <= 1).sum()
within_5km = (distances <= 0.5).sum()

print(f"\nCentroids within 1m of an ERA5 location: {within_5km} ({(within_5km/len(distances))*100:.2f}%)")
print(f"Centroids within 100m of an ERA5 location: {within_30km} ({(within_30km/len(distances))*100:.2f}%)")

=== Checking Spatial Overlay using final_df Centroids ===
final_centroids CRS: Asia_North_Albers_Equal_Area_Conic


NameError: name 'unique_geos_gdf' is not defined

In [58]:
import polars as pl
import pandas as pd
import geopandas as gpd
import duckdb
from shapely import wkt
import time

print("=== Phase 3: Temporal and Spatial Joining of ERA5 Data ===")

# 1. Fetch exactly the `.geo` string mappings for our final_df centroids
print("1. Establishing Spatial Link to exact ERA5 strings...")
unique_geos_df = pl.scan_parquet(r'era5_data\era5_master_dataset_wkt.parquet').select(pl.col(".geo")).unique().collect()

unique_geos_gdf = gpd.GeoDataFrame(
    {
        '.geo': unique_geos_df['.geo'].to_list(),
        'geometry': [wkt.loads(x) if pd.notna(x) else None for x in unique_geos_df['.geo'].to_list()]
    },
    crs='EPSG:4326'
)

# Use centroids of final_gdf to match
final_centroids_gdf = final_gdf.copy()
final_centroids_gdf['geometry'] = final_centroids_gdf.geometry.centroid

# Match geometries via nearest neighbor in the projected CRS
unique_geos_reproj = unique_geos_gdf.to_crs(final_centroids_gdf.crs)
nearest_join = gpd.sjoin_nearest(
    final_centroids_gdf.reset_index(names='original_index'), 
    unique_geos_reproj, 
    how='inner'
).drop_duplicates(subset=['original_index'])

# Create our precise lookup table
query_targets = nearest_join[['original_index', 'GLAKE_ID', 'event_date', '.geo']].copy()
# Ensure event_date is standard datetime
query_targets['event_date'] = pd.to_datetime(query_targets['event_date']).dt.normalize()

print(f"Mapped {len(query_targets)} unique final_df rows to ERA5 geo hashes.")

# 2. Execute massive aggregated JOIN via DuckDB
print("\n2. Executing Temporal Aggregations in DuckDB...")
start_time = time.time()

# Note: "3-day" means day of event + 2 days prior = 2 days interval.
# "7-day" = 6 days interval, "15-day" = 14 days interval, "30-day" = 29 days interval.
# Note: The time column in ERA5 data is called 'date'

query = """
SELECT 
    q.original_index,
    
    -- 3 DAY WINDOW (Event day + 2 prior days)
    SUM(CASE WHEN CAST(e.date AS DATE) BETWEEN q.event_date - INTERVAL 2 DAYS AND q.event_date THEN e.total_precipitation_sum END) AS total_precipitation_sum_3d,
    SUM(CASE WHEN CAST(e.date AS DATE) BETWEEN q.event_date - INTERVAL 2 DAYS AND q.event_date THEN e.snowmelt_sum END) AS snowmelt_sum_3d,
    SUM(CASE WHEN CAST(e.date AS DATE) BETWEEN q.event_date - INTERVAL 2 DAYS AND q.event_date THEN e.surface_runoff_sum END) AS surface_runoff_sum_3d,
    AVG(CASE WHEN CAST(e.date AS DATE) BETWEEN q.event_date - INTERVAL 2 DAYS AND q.event_date THEN e.surface_pressure END) AS surface_pressure_mean_3d,
    AVG(CASE WHEN CAST(e.date AS DATE) BETWEEN q.event_date - INTERVAL 2 DAYS AND q.event_date THEN e.dewpoint_temperature_2m END) AS dewpoint_temperature_2m_mean_3d,

    -- 7 DAY WINDOW (Event day + 6 prior days)
    SUM(CASE WHEN CAST(e.date AS DATE) BETWEEN q.event_date - INTERVAL 6 DAYS AND q.event_date THEN e.total_precipitation_sum END) AS total_precipitation_sum_7d,
    SUM(CASE WHEN CAST(e.date AS DATE) BETWEEN q.event_date - INTERVAL 6 DAYS AND q.event_date THEN e.snowmelt_sum END) AS snowmelt_sum_7d,
    SUM(CASE WHEN CAST(e.date AS DATE) BETWEEN q.event_date - INTERVAL 6 DAYS AND q.event_date THEN e.surface_runoff_sum END) AS surface_runoff_sum_7d,
    SUM(CASE WHEN CAST(e.date AS DATE) BETWEEN q.event_date - INTERVAL 6 DAYS AND q.event_date THEN e.sub_surface_runoff_sum END) AS sub_surface_runoff_sum_7d,
    SUM(CASE WHEN CAST(e.date AS DATE) BETWEEN q.event_date - INTERVAL 6 DAYS AND q.event_date THEN e.snowfall_sum END) AS snowfall_sum_7d,
    AVG(CASE WHEN CAST(e.date AS DATE) BETWEEN q.event_date - INTERVAL 6 DAYS AND q.event_date THEN e.temperature_2m END) AS temperature_2m_mean_7d,
    AVG(CASE WHEN CAST(e.date AS DATE) BETWEEN q.event_date - INTERVAL 6 DAYS AND q.event_date THEN e.dewpoint_temperature_2m END) AS dewpoint_temperature_2m_mean_7d,
    AVG(CASE WHEN CAST(e.date AS DATE) BETWEEN q.event_date - INTERVAL 6 DAYS AND q.event_date THEN e.snow_depth_water_equivalent END) AS snow_depth_water_equivalent_mean_7d,
    AVG(CASE WHEN CAST(e.date AS DATE) BETWEEN q.event_date - INTERVAL 6 DAYS AND q.event_date THEN e.surface_pressure END) AS surface_pressure_mean_7d,

    -- 15 DAY WINDOW (Event day + 14 prior days)
    SUM(CASE WHEN CAST(e.date AS DATE) BETWEEN q.event_date - INTERVAL 14 DAYS AND q.event_date THEN e.total_precipitation_sum END) AS total_precipitation_sum_15d,
    SUM(CASE WHEN CAST(e.date AS DATE) BETWEEN q.event_date - INTERVAL 14 DAYS AND q.event_date THEN e.snowmelt_sum END) AS snowmelt_sum_15d,
    SUM(CASE WHEN CAST(e.date AS DATE) BETWEEN q.event_date - INTERVAL 14 DAYS AND q.event_date THEN e.surface_runoff_sum END) AS surface_runoff_sum_15d,
    SUM(CASE WHEN CAST(e.date AS DATE) BETWEEN q.event_date - INTERVAL 14 DAYS AND q.event_date THEN e.sub_surface_runoff_sum END) AS sub_surface_runoff_sum_15d,
    SUM(CASE WHEN CAST(e.date AS DATE) BETWEEN q.event_date - INTERVAL 14 DAYS AND q.event_date THEN e.snowfall_sum END) AS snowfall_sum_15d,
    AVG(CASE WHEN CAST(e.date AS DATE) BETWEEN q.event_date - INTERVAL 14 DAYS AND q.event_date THEN e.temperature_2m END) AS temperature_2m_mean_15d,
    AVG(CASE WHEN CAST(e.date AS DATE) BETWEEN q.event_date - INTERVAL 14 DAYS AND q.event_date THEN e.dewpoint_temperature_2m END) AS dewpoint_temperature_2m_mean_15d,
    AVG(CASE WHEN CAST(e.date AS DATE) BETWEEN q.event_date - INTERVAL 14 DAYS AND q.event_date THEN e.snow_depth_water_equivalent END) AS snow_depth_water_equivalent_mean_15d,

    -- 30 DAY WINDOW (Event day + 29 prior days)
    SUM(CASE WHEN CAST(e.date AS DATE) BETWEEN q.event_date - INTERVAL 29 DAYS AND q.event_date THEN e.total_precipitation_sum END) AS total_precipitation_sum_30d,
    SUM(CASE WHEN CAST(e.date AS DATE) BETWEEN q.event_date - INTERVAL 29 DAYS AND q.event_date THEN e.snowmelt_sum END) AS snowmelt_sum_30d,
    SUM(CASE WHEN CAST(e.date AS DATE) BETWEEN q.event_date - INTERVAL 29 DAYS AND q.event_date THEN e.sub_surface_runoff_sum END) AS sub_surface_runoff_sum_30d,
    SUM(CASE WHEN CAST(e.date AS DATE) BETWEEN q.event_date - INTERVAL 29 DAYS AND q.event_date THEN e.surface_runoff_sum END) AS surface_runoff_sum_30d,
    SUM(CASE WHEN CAST(e.date AS DATE) BETWEEN q.event_date - INTERVAL 29 DAYS AND q.event_date THEN e.snowfall_sum END) AS snowfall_sum_30d,
    AVG(CASE WHEN CAST(e.date AS DATE) BETWEEN q.event_date - INTERVAL 29 DAYS AND q.event_date THEN e.temperature_2m END) AS temperature_2m_mean_30d,
    AVG(CASE WHEN CAST(e.date AS DATE) BETWEEN q.event_date - INTERVAL 29 DAYS AND q.event_date THEN e.snow_depth_water_equivalent END) AS snow_depth_water_equivalent_mean_30d,

    -- 90 DAY WINDOW (Event day + 89 prior days)
    SUM(CASE WHEN CAST(e.date AS DATE) BETWEEN q.event_date - INTERVAL 89 DAYS AND q.event_date THEN e.total_precipitation_sum END) AS total_precipitation_sum_90d,
    SUM(CASE WHEN CAST(e.date AS DATE) BETWEEN q.event_date - INTERVAL 89 DAYS AND q.event_date THEN e.snowmelt_sum END) AS snowmelt_sum_90d,
    SUM(CASE WHEN CAST(e.date AS DATE) BETWEEN q.event_date - INTERVAL 89 DAYS AND q.event_date THEN e.sub_surface_runoff_sum END) AS sub_surface_runoff_sum_90d,
    SUM(CASE WHEN CAST(e.date AS DATE) BETWEEN q.event_date - INTERVAL 89 DAYS AND q.event_date THEN e.snowfall_sum END) AS snowfall_sum_90d,
    AVG(CASE WHEN CAST(e.date AS DATE) BETWEEN q.event_date - INTERVAL 89 DAYS AND q.event_date THEN e.temperature_2m END) AS temperature_2m_mean_90d,
    AVG(CASE WHEN CAST(e.date AS DATE) BETWEEN q.event_date - INTERVAL 89 DAYS AND q.event_date THEN e.snow_depth_water_equivalent END) AS snow_depth_water_equivalent_mean_90d

FROM query_targets q
LEFT JOIN read_parquet('era5_data/era5_master_dataset_wkt.parquet') e 
    ON q.".geo" = e.".geo"
    -- To make this extremely fast, load era5 rows within 90 days of the target
    AND CAST(e.date AS DATE) BETWEEN q.event_date - INTERVAL 89 DAYS AND q.event_date
GROUP BY q.original_index
ORDER BY q.original_index
"""

con = duckdb.connect()
# Execute query and fetch as pandas DataFrame
weather_features_df = con.execute(query).df()
con.close()

end_time = time.time()
print(f"Weather dataset built in {round(end_time - start_time, 2)} seconds!")

# 3. Merge back with our final dataset
print("\n3. Merging weather features to the final_df dataset...")
final_df_enriched = final_df.copy().reset_index(names='original_index')
final_df_enriched = final_df_enriched.merge(weather_features_df, on='original_index', how='left')
final_df_enriched = final_df_enriched.drop(columns=['original_index'])

# Preview
print("\nShape of new enriched dataset:", final_df_enriched.shape)
print("Preview of new weather columns:")
display(final_df_enriched.filter(regex='_3d|_7d|_15d|_30d|_90d').head())

=== Phase 3: Temporal and Spatial Joining of ERA5 Data ===
1. Establishing Spatial Link to exact ERA5 strings...
Mapped 1500 unique final_df rows to ERA5 geo hashes.

2. Executing Temporal Aggregations in DuckDB...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Weather dataset built in 6.76 seconds!

3. Merging weather features to the final_df dataset...

Shape of new enriched dataset: (1500, 134)
Preview of new weather columns:


,total_precipitation_sum_3d,snowmelt_sum_3d,surface_runoff_sum_3d,surface_pressure_mean_3d,dewpoint_temperature_2m_mean_3d,total_precipitation_sum_7d,snowmelt_sum_7d,surface_runoff_sum_7d,sub_surface_runoff_sum_7d,snowfall_sum_7d,...,surface_runoff_sum_30d,snowfall_sum_30d,temperature_2m_mean_30d,snow_depth_water_equivalent_mean_30d,total_precipitation_sum_90d,snowmelt_sum_90d,sub_surface_runoff_sum_90d,snowfall_sum_90d,temperature_2m_mean_90d,snow_depth_water_equivalent_mean_90d
0,0.026047,0.021394,4.420599e-02,56492.459473,275.505410,0.061245,0.049220,0.099433,0.002348,5.509667e-03,...,0.522622,0.065249,276.327895,9.549043,0.698928,0.308501,0.019583,0.128796,272.401097,9.553094
1,0.000732,0.000948,5.589426e-05,55208.552192,259.001160,0.005474,0.002518,0.000154,0.017334,5.363278e-03,...,0.004337,0.062266,268.792036,0.007444,0.403398,0.101967,0.268727,0.111735,273.982925,0.003107
2,0.006044,0.000000,5.960464e-08,60764.268772,266.609725,0.038481,0.000370,0.000029,0.011166,3.781746e-02,...,0.001295,0.079309,267.246341,0.174779,0.683646,0.144269,0.398808,0.353274,273.070056,0.077683
3,0.000043,0.000071,1.148880e-05,60692.870660,267.463428,0.001183,0.003167,0.000522,0.000306,9.369105e-07,...,0.010742,0.022624,278.277868,5.574749,0.167095,0.432637,0.000350,0.057319,277.328678,5.719947
4,0.006868,0.010475,1.408786e-03,54024.399197,274.312458,0.026669,0.014506,0.002599,0.000007,1.405212e-02,...,0.040250,0.037919,275.699652,0.008749,0.119283,0.381335,0.000007,0.079034,270.555411,0.203320


In [ ]:
gdf.to_file(r'model_data\Final data\stage_2_final_dataset5.gpkg', driver='GPKG')

: 

In [2]:
import geopandas as gpd
gdf = gpd.read_file(r'model_data\Final data\stage_2_final_dataset4.gpkg')


In [3]:
# Climate Anomalies
gdf['precip_anomaly_7d_90d'] = gdf['total_precipitation_sum_7d'] / (gdf['total_precipitation_sum_90d'] / 90 * 7 + 1e-5)
gdf['snowmelt_anomaly_7d_30d'] = gdf['snowmelt_sum_7d'] / (gdf['snowmelt_sum_30d'] / 30 * 7 + 1e-5)
gdf['temp_spike_7d_over_30d'] = gdf['temperature_2m_mean_7d'] - gdf['temperature_2m_mean_30d']
gdf['runoff_shock'] = gdf['surface_runoff_sum_3d'] / (gdf['surface_runoff_sum_30d'] / 10 + 1e-5)
# Physics Proxies
gdf['dam_instability_index'] = (gdf['dam_height_m'] * gdf['dam_slope_deg']) / (gdf['dam_width_m'] + 1)
gdf['ice_avalanche_calving_proxy'] = gdf['v_mean_2020'] * gdf['contact_m']
gdf['hydrostatic_pressure_proxy'] = gdf['volume_m3'] * gdf['max_depth_m']
# Hydrological Loading
gdf['watershed_to_lake_area_ratio'] = gdf['watershed_area_km2'] / (gdf['area_2020_km2'] + 1e-5)
# How much precip is entering the lake relative to its volume?
gdf['precip_loading_factor'] = (gdf['total_precipitation_sum_7d'] * gdf['watershed_area_km2']) / (gdf['volume_m3'] + 1e-5)


In [4]:
# Melt-dominated vs rain-dominated events
gdf['snowmelt_fraction_7d'] = (
    gdf['snowmelt_sum_7d'] /
    (gdf['snowmelt_sum_7d'] + 
     gdf['total_precipitation_sum_7d'] + 1e-8)
)
gdf['melt_acceleration'] = (
    gdf['snowmelt_sum_7d'] - 
    gdf['snowmelt_sum_30d'] / 4
)
gdf['melt_heat_index'] = (
    gdf['snowmelt_sum_7d'] *
    gdf['temperature_2m_mean_7d']
)